<a href="https://colab.research.google.com/github/magnogomes874/MVP/blob/main/MVP_Analise_Risco_Custo_Tempo_DataCenter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# MVP — Análise de Risco, Custo e Tempo para Implantação de Nova Unidade de Data Center

**Objetivo:** apoiar a decisão de implantação de uma nova unidade de uma empresa de armazenamento/controle de dados em nuvem.

O MVP combina:

- Base pública do Kaggle sobre operação energética de data center;
- análise exploratória;
- modelo de Machine Learning para estimar consumo de energia;
- projeção de OPEX e CAPEX;
- estimativa de prazo;
- simulação de Monte Carlo;
- índice de risco de 0 a 100;
- comparação entre cenários/localizações;
- recomendação automática de **IMPLEMENTAR / REVISAR / NÃO IMPLEMENTAR**.

> **Importante:** a base do Kaggle fornece dados operacionais/energéticos. Custos de implantação, prazo, preço de energia, capacidade e demais premissas de negócio são parâmetros do cenário e podem ser substituídos pelos dados reais da empresa.



## 1. Base utilizada

O projeto utiliza o dataset **Data Center Cold Source Control Dataset**, do Kaggle. Ele contém 3.498 registros horários de operação de um sistema de refrigeração de data center, incluindo carga dos servidores, temperaturas, consumo de energia, uso de chillers/AHUs, custo de energia e desvio de temperatura.

Fonte: Kaggle — Data Center Cold Source Control Dataset.


In [ ]:

# Instalação/importação das bibliotecas
!pip -q install kagglehub openpyxl

import os
import re
import json
import math
import glob
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

SEED = 42
np.random.seed(SEED)

print("Ambiente preparado.")


In [ ]:

# 2. Download automático da base do Kaggle
# O código tenta baixar sem exigir upload manual de arquivo.

import requests, zipfile, io, os, glob

KAGGLE_DATASET = "programmer3/data-center-cold-source-control-dataset"
DATA_DIR = Path("/content/kaggle_data")
DATA_DIR.mkdir(exist_ok=True)

url = f"https://www.kaggle.com/api/v1/datasets/download/{KAGGLE_DATASET}"

try:
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    z = zipfile.ZipFile(io.BytesIO(response.content))
    z.extractall(DATA_DIR)
    print("Download concluído.")
    print("Arquivos:", [p.name for p in DATA_DIR.rglob("*") if p.is_file()])
except Exception as e:
    print("Não foi possível baixar automaticamente:", e)
    print("No Colab, baixe a base no Kaggle e faça upload do CSV para /content/kaggle_data.")


In [ ]:

# 3. Localização e leitura do CSV

csv_files = list(DATA_DIR.rglob("*.csv"))

if not csv_files:
    raise FileNotFoundError(
        "Nenhum CSV encontrado. Faça upload do arquivo CSV do dataset para /content/kaggle_data e execute esta célula novamente."
    )

csv_path = csv_files[0]
df = pd.read_csv(csv_path)

print("Arquivo:", csv_path)
print("Dimensões:", df.shape)
display(df.head())
print("\nColunas:")
print(df.columns.tolist())


In [ ]:

# 4. Padronização automática dos nomes das colunas

def normalize_col(c):
    c = str(c).strip().lower()
    c = re.sub(r"[^a-z0-9]+", "_", c)
    return c.strip("_")

df.columns = [normalize_col(c) for c in df.columns]

print(df.columns.tolist())
display(df.describe(include="all").T)


In [ ]:

# 5. Identificação automática das principais variáveis

print("Tipos:")
display(df.dtypes.to_frame("tipo"))

print("\nValores ausentes:")
display(df.isna().sum().sort_values(ascending=False).to_frame("ausentes"))

# Detecta possíveis colunas numéricas
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
print("\nColunas numéricas:", numeric_cols)


## 6. Análise exploratória — relação entre carga, temperatura e consumo

In [ ]:

# Localiza a variável de consumo de energia/potência
def find_col(patterns):
    for p in patterns:
        for c in df.columns:
            if p in c:
                return c
    return None

power_col = find_col(["power_consumption", "power", "energy_consumption", "energy"])
workload_col = find_col(["server_workload", "workload", "load"])
ambient_col = find_col(["ambient_temperature", "ambient_temp"])
inlet_col = find_col(["inlet_temperature", "inlet_temp"])
outlet_col = find_col(["outlet_temperature", "outlet_temp"])

print("Potência/energia:", power_col)
print("Carga:", workload_col)
print("Temperatura ambiente:", ambient_col)
print("Temperatura entrada:", inlet_col)
print("Temperatura saída:", outlet_col)


In [ ]:

# Gráfico carga x consumo
if power_col and workload_col:
    plt.figure(figsize=(9,5))
    plt.scatter(df[workload_col], df[power_col], alpha=0.25)
    plt.xlabel("Carga do servidor (%)")
    plt.ylabel("Consumo de energia/potência")
    plt.title("Relação entre carga e consumo")
    plt.grid(alpha=0.2)
    plt.show()
else:
    print("Não foi possível localizar automaticamente as colunas de carga e consumo.")


In [ ]:

# 7. Modelo de Machine Learning para estimar consumo

if power_col is None:
    raise ValueError("A coluna de consumo não foi identificada.")

target = power_col

# Usamos somente variáveis numéricas como entrada para tornar o notebook robusto.
features = [c for c in numeric_cols if c != target]

if len(features) < 2:
    raise ValueError("Poucas variáveis numéricas disponíveis para treinar o modelo.")

X = df[features].copy()
y = pd.to_numeric(df[target], errors="coerce")

valid = y.notna()
X = X.loc[valid]
y = y.loc[valid]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED
)

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("rf", RandomForestRegressor(
        n_estimators=300,
        max_depth=14,
        min_samples_leaf=3,
        random_state=SEED,
        n_jobs=-1
    ))
])

model.fit(X_train, y_train)
pred = model.predict(X_test)

mae = mean_absolute_error(y_test, pred)
rmse = mean_squared_error(y_test, pred) ** 0.5
r2 = r2_score(y_test, pred)

print(f"MAE:  {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²:   {r2:.4f}")


In [ ]:

# 8. Importância das variáveis

rf = model.named_steps["rf"]

importance = pd.DataFrame({
    "variavel": features,
    "importancia": rf.feature_importances_
}).sort_values("importancia", ascending=False)

display(importance)

plt.figure(figsize=(9,5))
plt.barh(importance["variavel"].head(10)[::-1], importance["importancia"].head(10)[::-1])
plt.xlabel("Importância")
plt.title("Principais fatores associados ao consumo")
plt.grid(axis="x", alpha=0.2)
plt.show()



# 9. Motor de decisão da nova unidade

A partir daqui o MVP deixa de depender exclusivamente da base histórica.

O usuário informa as características da nova unidade:

- capacidade de servidores;
- utilização média;
- potência média por servidor;
- PUE;
- preço da energia;
- investimento inicial;
- custo de obra;
- custo de equipamentos;
- custo anual de pessoal;
- custo anual de manutenção;
- prazo-base;
- complexidade;
- infraestrutura;
- risco regulatório;
- risco de energia;
- demanda esperada.

O sistema transforma essas entradas em custo, prazo, risco e viabilidade.


In [ ]:

# 10. Parâmetros do projeto — ALTERE AQUI

cenario = {
    "nome": "Nova Unidade - Cenário Base",
    "servidores": 5000,
    "utilizacao_media": 0.65,
    "potencia_kw_por_servidor": 0.35,
    "pue": 1.35,
    "preco_energia_kwh": 0.85,

    # CAPEX
    "terreno_obra": 8_000_000,
    "servidores_equipamentos": 7_500_000,
    "rede_seguranca": 2_000_000,
    "instalacao_comissionamento": 1_500_000,

    # OPEX anual
    "pessoal_anual": 1_200_000,
    "manutencao_anual": 900_000,
    "conectividade_anual": 600_000,
    "outros_opex_anual": 400_000,

    # Prazo
    "prazo_base_meses": 14,

    # Riscos informados — 0 a 100
    "risco_regulatorio": 25,
    "risco_infraestrutura": 20,
    "risco_energia": 30,
    "risco_fornecedores": 25,
    "risco_demanda": 20,
    "risco_operacional": 15,

    # Receita/benefício anual estimado
    "beneficio_anual": 7_000_000
}

pd.DataFrame([cenario]).T.rename(columns={0:"valor"})


In [ ]:

# 11. Funções financeiras e operacionais

def calcular_cenario(c):
    servidores = c["servidores"]
    utilizacao = c["utilizacao_media"]
    potencia = c["potencia_kw_por_servidor"]
    pue = c["pue"]
    preco = c["preco_energia_kwh"]

    # Potência IT média
    potencia_it_kw = servidores * potencia * utilizacao

    # Potência total considerando PUE
    potencia_total_kw = potencia_it_kw * pue

    # Energia anual
    energia_anual_kwh = potencia_total_kw * 24 * 365

    # Custo energético
    energia_anual = energia_anual_kwh * preco

    capex = (
        c["terreno_obra"]
        + c["servidores_equipamentos"]
        + c["rede_seguranca"]
        + c["instalacao_comissionamento"]
    )

    opex_anual = (
        energia_anual
        + c["pessoal_anual"]
        + c["manutencao_anual"]
        + c["conectividade_anual"]
        + c["outros_opex_anual"]
    )

    beneficio = c["beneficio_anual"]
    payback = capex / max(beneficio - opex_anual, 1)

    # Risco ponderado
    pesos = {
        "risco_regulatorio": 0.15,
        "risco_infraestrutura": 0.20,
        "risco_energia": 0.20,
        "risco_fornecedores": 0.15,
        "risco_demanda": 0.15,
        "risco_operacional": 0.15
    }

    risco = sum(c[k] * peso for k, peso in pesos.items())

    # Classificação
    if risco <= 30:
        classe_risco = "BAIXO"
    elif risco <= 60:
        classe_risco = "MÉDIO"
    elif risco <= 80:
        classe_risco = "ALTO"
    else:
        classe_risco = "CRÍTICO"

    # Regra de decisão
    if risco <= 40 and payback <= 6:
        decisao = "IMPLEMENTAR"
    elif risco <= 65 and payback <= 8:
        decisao = "REVISAR PROJETO"
    else:
        decisao = "NÃO IMPLEMENTAR"

    return {
        "potencia_it_kw": potencia_it_kw,
        "potencia_total_kw": potencia_total_kw,
        "energia_anual_kwh": energia_anual_kwh,
        "custo_energia_anual": energia_anual,
        "capex": capex,
        "opex_anual": opex_anual,
        "resultado_operacional_anual": beneficio - opex_anual,
        "payback_anos": payback,
        "risco": risco,
        "classe_risco": classe_risco,
        "prazo_meses": c["prazo_base_meses"],
        "decisao": decisao
    }

resultado = calcular_cenario(cenario)
display(pd.DataFrame([resultado]).T.rename(columns={0:"resultado"}))


In [ ]:

# 12. Dashboard executivo

def moeda(x):
    return f"R$ {x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

print("="*65)
print("        MVP — DECISÃO DE EXPANSÃO DE NOVA UNIDADE")
print("="*65)
print(f"Projeto:              {cenario['nome']}")
print(f"CAPEX:                {moeda(resultado['capex'])}")
print(f"OPEX anual:           {moeda(resultado['opex_anual'])}")
print(f"Custo energia/ano:    {moeda(resultado['custo_energia_anual'])}")
print(f"Potência total:       {resultado['potencia_total_kw']:,.2f} kW")
print(f"Prazo base:            {resultado['prazo_meses']:.1f} meses")
print(f"Payback:              {resultado['payback_anos']:.2f} anos")
print(f"Risco:                {resultado['risco']:.1f}/100 — {resultado['classe_risco']}")
print("-"*65)
print(f"RECOMENDAÇÃO:         {resultado['decisao']}")
print("="*65)


## 13. Simulação de Monte Carlo — custo, prazo e risco

In [ ]:

# A simulação introduz incerteza nos principais fatores.
# Isso é importante porque uma decisão de expansão não deve considerar somente um valor pontual.

def monte_carlo(c, n=10000, seed=42):
    rng = np.random.default_rng(seed)

    # Incertezas
    fator_capex = rng.triangular(0.90, 1.05, 1.25, n)
    fator_energia = rng.triangular(0.85, 1.00, 1.35, n)
    fator_opex = rng.triangular(0.90, 1.05, 1.20, n)
    fator_prazo = rng.triangular(0.85, 1.10, 1.45, n)
    fator_beneficio = rng.triangular(0.75, 1.00, 1.15, n)

    base = calcular_cenario(c)

    capex = base["capex"] * fator_capex

    energia = base["custo_energia_anual"] * fator_energia

    opex_sem_energia = base["opex_anual"] - base["custo_energia_anual"]
    opex = opex_sem_energia * fator_opex + energia

    beneficio = c["beneficio_anual"] * fator_beneficio

    resultado_op = beneficio - opex
    payback = capex / np.maximum(resultado_op, 1)

    prazo = c["prazo_base_meses"] * fator_prazo

    # risco econômico derivado da dispersão dos resultados
    risco_sim = (
        0.35 * np.clip((capex / base["capex"] - 1) * 100 + base["risco"], 0, 100)
        + 0.25 * np.clip((prazo / c["prazo_base_meses"] - 1) * 100 + base["risco"], 0, 100)
        + 0.20 * base["risco"]
        + 0.20 * np.clip((1 - resultado_op / max(c["beneficio_anual"], 1)) * 100, 0, 100)
    )

    return pd.DataFrame({
        "capex": capex,
        "opex_anual": opex,
        "beneficio_anual": beneficio,
        "resultado_operacional": resultado_op,
        "payback_anos": payback,
        "prazo_meses": prazo,
        "risco": np.clip(risco_sim, 0, 100)
    })

sim = monte_carlo(cenario)

display(sim.describe(percentiles=[.05,.25,.50,.75,.95]).T)


In [ ]:

# 14. Probabilidades de sucesso

prob_payback = (sim["payback_anos"] <= 6).mean()
prob_prazo = (sim["prazo_meses"] <= 18).mean()
prob_risco = (sim["risco"] <= 50).mean()

prob_sucesso = (
    (sim["payback_anos"] <= 6)
    & (sim["prazo_meses"] <= 18)
    & (sim["risco"] <= 50)
).mean()

print(f"Probabilidade de payback <= 6 anos: {prob_payback:.1%}")
print(f"Probabilidade de prazo <= 18 meses: {prob_prazo:.1%}")
print(f"Probabilidade de risco <= 50:       {prob_risco:.1%}")
print(f"Probabilidade conjunta de sucesso:  {prob_sucesso:.1%}")


In [ ]:

# 15. Distribuições de risco, prazo e payback

fig = plt.figure(figsize=(9,5))
plt.hist(sim["payback_anos"], bins=50)
plt.axvline(6, linestyle="--", label="Meta: 6 anos")
plt.xlabel("Payback (anos)")
plt.ylabel("Frequência")
plt.title("Distribuição do Payback — Monte Carlo")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

fig = plt.figure(figsize=(9,5))
plt.hist(sim["prazo_meses"], bins=50)
plt.axvline(18, linestyle="--", label="Limite: 18 meses")
plt.xlabel("Prazo (meses)")
plt.ylabel("Frequência")
plt.title("Distribuição do Prazo — Monte Carlo")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

fig = plt.figure(figsize=(9,5))
plt.hist(sim["risco"], bins=50)
plt.axvline(50, linestyle="--", label="Limite: risco 50")
plt.xlabel("Índice de risco")
plt.ylabel("Frequência")
plt.title("Distribuição do Risco — Monte Carlo")
plt.legend()
plt.grid(alpha=0.2)
plt.show()


## 16. Comparação de possíveis localidades/cenários

In [ ]:

# Edite os cenários para representar cidades/regiões diferentes.
# Os valores abaixo são EXEMPLOS para demonstrar o funcionamento do MVP.

cenarios = [
    {
        "local": "Local A",
        "capex_fator": 1.00,
        "energia": 0.85,
        "prazo": 14,
        "risco": 25,
        "beneficio": 7_000_000
    },
    {
        "local": "Local B",
        "capex_fator": 0.90,
        "energia": 1.05,
        "prazo": 11,
        "risco": 48,
        "beneficio": 7_200_000
    },
    {
        "local": "Local C",
        "capex_fator": 1.15,
        "energia": 0.70,
        "prazo": 18,
        "risco": 18,
        "beneficio": 7_600_000
    }
]

comparacao = []

for s in cenarios:
    c = cenario.copy()
    c["nome"] = s["local"]
    c["terreno_obra"] *= s["capex_fator"]
    c["servidores_equipamentos"] *= s["capex_fator"]
    c["rede_seguranca"] *= s["capex_fator"]
    c["instalacao_comissionamento"] *= s["capex_fator"]
    c["preco_energia_kwh"] = s["energia"]
    c["prazo_base_meses"] = s["prazo"]
    c["beneficio_anual"] = s["beneficio"]

    # distribui o risco geral entre os fatores
    for k in [
        "risco_regulatorio", "risco_infraestrutura", "risco_energia",
        "risco_fornecedores", "risco_demanda", "risco_operacional"
    ]:
        c[k] = s["risco"]

    r = calcular_cenario(c)

    comparacao.append({
        "Local": s["local"],
        "CAPEX": r["capex"],
        "OPEX anual": r["opex_anual"],
        "Prazo (meses)": r["prazo_meses"],
        "Payback (anos)": r["payback_anos"],
        "Risco": r["risco"],
        "Classe": r["classe_risco"],
        "Decisão": r["decisao"]
    })

comparacao_df = pd.DataFrame(comparacao)
display(comparacao_df)


In [ ]:

# 17. Ranking automático

# Score de viabilidade:
# quanto menor o risco, menor o payback e menor o prazo, melhor.
rank = comparacao_df.copy()

def normalizar_inverso(s):
    return 1 - (s - s.min()) / (s.max() - s.min() + 1e-9)

rank["score_risco"] = normalizar_inverso(rank["Risco"])
rank["score_payback"] = normalizar_inverso(rank["Payback (anos)"])
rank["score_prazo"] = normalizar_inverso(rank["Prazo (meses)"])

rank["score_final"] = (
    0.40 * rank["score_risco"]
    + 0.35 * rank["score_payback"]
    + 0.25 * rank["score_prazo"]
) * 100

rank = rank.sort_values("score_final", ascending=False)

display(rank[[
    "Local", "CAPEX", "Prazo (meses)", "Payback (anos)",
    "Risco", "Decisão", "score_final"
]])



## 18. Simulador interativo

A célula abaixo permite testar diferentes investimentos, capacidade, preço de energia e riscos sem alterar manualmente todas as fórmulas.


In [ ]:

# 18.1 Simulador simples com função
def simular_nova_unidade(
    servidores=5000,
    utilizacao=0.65,
    preco_energia=0.85,
    capex=19_000_000,
    prazo=14,
    risco=30,
    beneficio=7_000_000
):
    c = cenario.copy()

    c["servidores"] = servidores
    c["utilizacao_media"] = utilizacao
    c["preco_energia_kwh"] = preco_energia
    c["prazo_base_meses"] = prazo
    c["beneficio_anual"] = beneficio

    # Ajusta os componentes de CAPEX proporcionalmente
    capex_original = (
        c["terreno_obra"] + c["servidores_equipamentos"]
        + c["rede_seguranca"] + c["instalacao_comissionamento"]
    )
    fator = capex / capex_original

    c["terreno_obra"] *= fator
    c["servidores_equipamentos"] *= fator
    c["rede_seguranca"] *= fator
    c["instalacao_comissionamento"] *= fator

    for k in [
        "risco_regulatorio", "risco_infraestrutura", "risco_energia",
        "risco_fornecedores", "risco_demanda", "risco_operacional"
    ]:
        c[k] = risco

    r = calcular_cenario(c)

    return pd.Series({
        "CAPEX": moeda(r["capex"]),
        "OPEX anual": moeda(r["opex_anual"]),
        "Energia anual": moeda(r["custo_energia_anual"]),
        "Prazo": f"{r['prazo_meses']:.1f} meses",
        "Payback": f"{r['payback_anos']:.2f} anos",
        "Risco": f"{r['risco']:.1f}/100 ({r['classe_risco']})",
        "Decisão": r["decisao"]
    })

display(simular_nova_unidade())



# 19. Exportação dos resultados

Os resultados podem ser exportados para Excel para apresentação à gestão.


In [ ]:

# Exporta os principais resultados
arquivo_excel = "/content/MVP_Analise_Expansao_DataCenter.xlsx"

with pd.ExcelWriter(arquivo_excel, engine="openpyxl") as writer:
    pd.DataFrame([cenario]).T.rename(columns={0:"valor"}).to_excel(
        writer, sheet_name="Premissas"
    )
    pd.DataFrame([resultado]).T.rename(columns={0:"valor"}).to_excel(
        writer, sheet_name="Resultado_Base"
    )
    sim.describe().T.to_excel(writer, sheet_name="Monte_Carlo")
    comparacao_df.to_excel(writer, sheet_name="Comparacao", index=False)
    rank.to_excel(writer, sheet_name="Ranking", index=False)

print("Arquivo gerado:", arquivo_excel)



# 20. Conclusão do MVP

O protótipo entrega uma estrutura de decisão em quatro níveis:

### 1 — Dados
Base histórica do Kaggle para entender o comportamento energético de uma operação de data center.

### 2 — Predição
Modelo Random Forest para relacionar características operacionais ao consumo.

### 3 — Planejamento
Cálculo de CAPEX, OPEX, energia, payback e prazo.

### 4 — Risco
Monte Carlo + matriz ponderada de riscos para estimar a probabilidade de o projeto atingir as metas.

## Como transformar este MVP em um projeto empresarial

1. Substituir as premissas fictícias pelos custos reais da empresa.
2. Adicionar dados reais de fornecedores, obras, energia e infraestrutura.
3. Criar histórico de projetos anteriores para treinar modelos de prazo e custo.
4. Incluir localização real e dados de energia/conectividade.
5. Criar cenários **otimista, base e pessimista**.
6. Adicionar aprovação por etapas: estudo → orçamento → implantação → operação.
7. Migrar o dashboard para uma aplicação web quando o modelo estiver validado.

**Resultado final esperado:** uma ferramenta que permita à gestão testar diferentes locais e configurações e responder, com dados, **quanto custa, quanto tempo leva, qual o risco e se vale a pena implantar a nova unidade**.
